<a href="https://colab.research.google.com/github/Arzuyasar/amazon-customer-review/blob/main/Faz3_NLP_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q vaderSentiment transformers sumy torch tqdm

import pandas as pd
import numpy as np
import os, gc, time
from tqdm.auto import tqdm
from IPython.display import display

from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
print("Hazır")

Mounted at /content/drive
Hazır
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hazır


In [3]:
df = pd.read_parquet(DRIVE_PATH + 'faz2_temiz_veri.parquet')
print(f"{len(df):,} satır yüklendi")

ORNEKLEM = 200_000
df_nlp   = df.head(ORNEKLEM).copy()
print(f"   Çalışılacak satır: {len(df_nlp):,} (geliştirme modu)")
print("   Final için ORNEKLEM değişkenini kaldır veya büyüt")

257,378 satır yüklendi
   Çalışılacak satır: 200,000 (geliştirme modu)
   Final için ORNEKLEM değişkenini kaldır veya büyüt


In [43]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def vader_skor_hesapla(text: str) -> float:

    return analyzer.polarity_scores(str(text))['compound']

print("VADER skoru hesaplanıyor...")
df_nlp['vader_compound'] = df_nlp['review_body'].apply(vader_skor_hesapla)

print(f"Tamamlandı")
print("\nVADER skoru dağılımı:")
print(df_nlp['vader_compound'].describe().round(3).to_string())

print("\nÖrnek yorumlar:")
print("─" * 70)
ornekler = df_nlp[['review_body','star_rating','vader_compound']].sample(5)
for _, row in ornekler.iterrows():
    print(f"Yorum: {str(row['review_body'])[:70]}...")
    print(f" {row['star_rating']}  VADER: {row['vader_compound']:+.3f}\n")

VADER skoru hesaplanıyor...
Tamamlandı

VADER skoru dağılımı:
count    200000.000
mean          0.468
std           0.513
min          -1.000
25%           0.128
50%           0.659
75%           0.872
max           1.000

Örnek yorumlar:
──────────────────────────────────────────────────────────────────────
Yorum: This was replacement for the identical item. Both have been excellent ...
 5  VADER: +0.735

Yorum: Its an OK multiplayer experience. I'm more of a roleplaying and strate...
 3  VADER: +0.557

Yorum: Bought these a bit over a year ago and let me tell you they suck! The ...
 1  VADER: -0.541

Yorum: I brought this for the gym and lets just say that if you go to the gym...
 1  VADER: -0.176

Yorum: It has a human voice warning that give distance in centimeters. No buz...
 3  VADER: -0.718



In [5]:
def problem_tespit(star_rating, vader_compound):
    yildiz_sinyal = int(star_rating <= 3)
    vader_sinyal  = int(vader_compound < -0.05)
    skor          = yildiz_sinyal * 0.6 + vader_sinyal * 0.4
    return int(skor >= 0.5)

df_nlp['problem_var'] = df_nlp.apply(
    lambda r: problem_tespit(r['star_rating'], r['vader_compound']), axis=1
)

print(f"Problem tespit sonuçları:")
print(f"  Problemli yorum : {df_nlp['problem_var'].sum():,}  (%{df_nlp['problem_var'].mean()*100:.1f})")
print(f"  Sorunsuz yorum  : {(1-df_nlp['problem_var']).sum():,}  (%{(1-df_nlp['problem_var'].mean())*100:.1f})")

Problem tespit sonuçları:
  Problemli yorum : 58,616  (%29.3)
  Sorunsuz yorum  : 141,384  (%70.7)


In [44]:
from transformers import pipeline as hf_pipeline

print("Model yükleniyor (ilk seferinde 1-2 dk sürer)...")
siniflandirici = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0,
)
print("Model hazır")

ETIKETLER = [
    'hardware defect or malfunction',
    'software bug or app issue',
    'shipping and delivery problem',
    'customer service complaint',
    'product design or usability issue',
    'positive review no problem',
]

TURKCE = {
    'hardware defect or malfunction':     'Teknik Destek',
    'software bug or app issue':          'Yazılım Ekibi',
    'shipping and delivery problem':      'Lojistik',
    'customer service complaint':         'Müşteri Hizmetleri',
    'product design or usability issue':  'Ürün Yönetimi',
    'positive review no problem':         'Arşiv (Olumlu)',
}

def departman_bul(metin: str) -> dict:

    sonuc    = siniflandirici(str(metin)[:512], ETIKETLER, multi_label=False)
    en_iyi   = sonuc['labels'][0]
    return {
        'departman_en': en_iyi,
        'departman_tr': TURKCE[en_iyi],
        'guven_skoru':  round(sonuc['scores'][0], 3),
    }

print("\nTest sonuçları:")
print("─" * 70)
test_metinler = [
    "The screen cracked after one week. Hardware quality is terrible.",
    "App crashes every time I open it. Please fix the software.",
    "Package arrived completely crushed. Box was destroyed during shipping.",
    "Customer service never replied to my 5 emails. Unacceptable.",
    "Absolutely love this product! Best purchase I made this year.",
]
for metin in test_metinler:
    sonuc = departman_bul(metin)
    print(f"Yorum    : {metin[:60]}...")
    print(f"Departman: {sonuc['departman_tr']}  (güven: {sonuc['guven_skoru']:.2f})\n")

Model yükleniyor (ilk seferinde 1-2 dk sürer)...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Model hazır

Test sonuçları:
──────────────────────────────────────────────────────────────────────
Yorum    : The screen cracked after one week. Hardware quality is terri...
Departman: Teknik Destek  (güven: 0.92)

Yorum    : App crashes every time I open it. Please fix the software....
Departman: Yazılım Ekibi  (güven: 0.90)

Yorum    : Package arrived completely crushed. Box was destroyed during...
Departman: Lojistik  (güven: 0.70)

Yorum    : Customer service never replied to my 5 emails. Unacceptable....
Departman: Müşteri Hizmetleri  (güven: 0.90)

Yorum    : Absolutely love this product! Best purchase I made this year...
Departman: Arşiv (Olumlu)  (güven: 0.98)



In [8]:

!pip install -q transformers torch tqdm

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

KURALLAR = {
    'Teknik Destek': [
        'broken', 'defect', 'malfunction', 'not working', 'stopped working',
        'hardware', 'damage', 'cracked', 'dead', 'faulty', 'repair',
        'doesnt work', "doesn't work", 'stopped', 'fell apart'
    ],
    'Yazılım Ekibi': [
        'software', 'app', 'crash', 'bug', 'update', 'glitch', 'freeze',
        'error', 'install', 'compatible', 'firmware', 'driver', 'reboot',
        'connection', 'bluetooth', 'wifi', 'sync', 'patch'
    ],
    'Lojistik': [
        'shipping', 'delivery', 'package', 'arrived', 'late', 'damaged box',
        'missing', 'lost', 'never received', 'wrong item', 'return',
        'packaging', 'shipped', 'courier', 'tracking', 'delay'
    ],
    'Müşteri Hizmetleri': [
        'customer service', 'support', 'refund', 'warranty', 'response',
        'replied', 'contact', 'ignored', 'rude', 'helpful', 'service',
        'representative', 'call center', 'exchange', 'complaint'
    ],
    'Ürün Yönetimi': [
        'design', 'quality', 'cheap', 'material', 'size', 'color',
        'uncomfortable', 'usability', 'confusing', 'misleading', 'overpriced',
        'build quality', 'poorly made', 'flimsy', 'look', 'feel'
    ],
}


def kural_ile_departman(metin: str) -> dict:
    metin_lower = str(metin).lower()
    skorlar = {}

    for dept, kelimeler in KURALLAR.items():
        eslesme = sum(1 for k in kelimeler if k in metin_lower)
        skorlar[dept] = eslesme

    en_iyi = max(skorlar, key=skorlar.get)
    max_skor = skorlar[en_iyi]

    if max_skor == 0:
        return {'departman_tr': 'Teknik Destek', 'guven_skoru': 0.25}

    guven = min(0.45 + max_skor * 0.1, 0.90)
    return {'departman_tr': en_iyi, 'guven_skoru': round(guven, 2)}

print("Kural tabanlı sınıflandırma başlıyor...")
problemli_idx = df_nlp[df_nlp['problem_var'] == 1].index
print(f"Toplam problemli yorum: {len(problemli_idx):,}")

tqdm.pandas(desc="Kural tabanlı")
sonuclar = df_nlp.loc[problemli_idx, 'review_body'].progress_apply(kural_ile_departman)
sonuc_df = pd.DataFrame(sonuclar.tolist(), index=problemli_idx)
df_nlp   = df_nlp.join(sonuc_df)

df_nlp.loc[df_nlp['problem_var'] == 0, 'departman_tr'] = 'Arşiv (Olumlu)'
df_nlp.loc[df_nlp['problem_var'] == 0, 'guven_skoru']  = 1.0

print(f"\nKural tabanlı tamamlandı")
print(f"\nDepartman dağılımı (henüz kural tabanlı):")
print(df_nlp['departman_tr'].value_counts().to_string())
print(f"\nGüven skoru dağılımı:")
print(df_nlp.loc[problemli_idx, 'guven_skoru'].describe().round(2).to_string())

Kural tabanlı sınıflandırma başlıyor...
Toplam problemli yorum: 58,616


Kural tabanlı:   0%|          | 0/58616 [00:00<?, ?it/s]


Kural tabanlı tamamlandı

Departman dağılımı (henüz kural tabanlı):
departman_tr
Arşiv (Olumlu)        141384
Teknik Destek          29422
Yazılım Ekibi          11317
Ürün Yönetimi          10302
Lojistik                5470
Müşteri Hizmetleri      2105

Güven skoru dağılımı:
count    58616.00
mean         0.46
std          0.18
min          0.25
25%          0.25
50%          0.55
75%          0.55
max          0.90


In [10]:
GUVEN_ESIK  = 0.45
MAX_ZEROSHT = 500

dusuk_guven_idx = df_nlp[
    (df_nlp['problem_var'] == 1) &
    (df_nlp['guven_skoru'] < GUVEN_ESIK)
].index

print(f"Düşük güvenli yorum sayısı : {len(dusuk_guven_idx):,}")
print(f"Zero-shot'a gidecek        : {min(len(dusuk_guven_idx), MAX_ZEROSHT):,}")
print(f"Toplam yorum sayısı       : {len(df_nlp):,}")
from transformers import pipeline as hf_pipeline

print("\nZero-shot modeli yükleniyor...")
print("(İlk seferinde ~800 MB indirir, sonraki çalışmalarda önbellekten gelir)")

siniflandirici = hf_pipeline(
    'zero-shot-classification',
    model='facebook/bart-large-mnli',
    device=0,
)
print("Model hazır")

ETIKETLER = [
    'hardware defect or malfunction',
    'software bug or app issue',
    'shipping and delivery problem',
    'customer service complaint',
    'product design or usability issue',
]
TURKCE = {
    'hardware defect or malfunction':    'Teknik Destek',
    'software bug or app issue':         'Yazılım Ekibi',
    'shipping and delivery problem':     'Lojistik',
    'customer service complaint':        'Müşteri Hizmetleri',
    'product design or usability issue': 'Ürün Yönetimi',
}

def zero_shot_departman(metin: str) -> dict:
    try:
        sonuc  = siniflandirici(str(metin)[:512], ETIKETLER, multi_label=False)
        en_iyi = sonuc['labels'][0]
        return {
            'departman_tr': TURKCE[en_iyi],
            'guven_skoru':  round(sonuc['scores'][0], 3),
        }
    except Exception:
        return {'departman_tr': 'Teknik Destek', 'guven_skoru': 0.3}

Düşük güvenli yorum sayısı : 22,926
Zero-shot'a gidecek        : 500
Toplam yorum sayısı       : 200,000

Zero-shot modeli yükleniyor...
(İlk seferinde ~800 MB indirir, sonraki çalışmalarda önbellekten gelir)


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Model hazır


In [12]:
hedef_idx = dusuk_guven_idx[:MAX_ZEROSHT]
print(f"\nZero-shot sınıflandırma başlıyor ({len(hedef_idx):,} yorum)...")

zero_sonuclar = []
BATCH = 16

for i in tqdm(range(0, len(hedef_idx), BATCH), desc="Zero-shot"):
    batch_idx = hedef_idx[i:i+BATCH]
    for idx in batch_idx:
        metin = df_nlp.loc[idx, 'review_body']
        zero_sonuclar.append(zero_shot_departman(metin))

zero_df = pd.DataFrame(zero_sonuclar, index=hedef_idx)
df_nlp.loc[hedef_idx, 'departman_tr'] = zero_df['departman_tr'].values
df_nlp.loc[hedef_idx, 'guven_skoru']  = zero_df['guven_skoru'].values

print("\nZero-shot doğrulama tamamlandı")


Zero-shot sınıflandırma başlıyor (500 yorum)...


Zero-shot:   0%|          | 0/32 [00:00<?, ?it/s]


Zero-shot doğrulama tamamlandı


In [31]:
print("── Sınıflandırma Tamamlandı ─────────────────────────────")
print(f"Toplam yorum        : {len(df_nlp):,}")
print(f"Problemli           : {df_nlp['problem_var'].sum():,}")
print(f"Sorunsuz (Arşiv)    : {(df_nlp['problem_var']==0).sum():,}")

print("\nDepartman dağılımı:")
print(df_nlp['departman_tr'].value_counts().to_string())

print("\nOrtalama güven skoru (departmana göre):")
print(
    df_nlp[df_nlp['problem_var']==1]
    .groupby('departman_tr')['guven_skoru']
    .mean().round(3)
    .sort_values(ascending=False)
    .to_string()
)

── Sınıflandırma Tamamlandı ─────────────────────────────
Toplam yorum        : 200,000
Problemli           : 58,616
Sorunsuz (Arşiv)    : 141,384

Departman dağılımı:
departman_tr
Arşiv (Olumlu)        141384
Teknik Destek          28992
Yazılım Ekibi          11330
Ürün Yönetimi          10695
Lojistik                5483
Müşteri Hizmetleri      2116

Ortalama güven skoru (departmana göre):
departman_tr
Müşteri Hizmetleri    0.624
Ürün Yönetimi         0.610
Yazılım Ekibi         0.589
Lojistik              0.584
Teknik Destek         0.325


In [35]:
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lsa import LsaSummarizer
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def ozet_cikar(metin: str, cumle_sayisi: int = 2) -> str:

    metin = str(metin).strip()

    if len(metin) < 100:
        return metin

    try:
        parser     = PlaintextParser.from_string(metin, Tokenizer('english'))
        summarizer = LsaSummarizer()
        ozet       = summarizer(parser.document, cumle_sayisi)
        sonuc      = ' '.join(str(c) for c in ozet)
        return sonuc if sonuc.strip() else metin[:200]
    except Exception:
        return metin[:200]

ornek_metin = df_nlp['review_body'].dropna().iloc[0]
print("Orijinal yorum:")
print(ornek_metin)
print(f"\nÖzet ({len(ornek_metin)} → {len(ozet_cikar(ornek_metin))} karakter):")
print(ozet_cikar(ornek_metin))

Orijinal yorum:
Used this for Elite Dangerous on my mac, an amazing joystick. I especially love that you can twist the stick for different movement bindings as well as move it in the normal way.

Özet (178 → 178 karakter):
Used this for Elite Dangerous on my mac, an amazing joystick. I especially love that you can twist the stick for different movement bindings as well as move it in the normal way.


In [36]:
print("Yorumlar özetleniyor...")
tqdm.pandas(desc="Özetleniyor")
df_nlp['ozet'] = df_nlp['review_body'].progress_apply(ozet_cikar)

print(f"Özetleme tamamlandı")
print(f"Ortalama orijinal uzunluk : {df_nlp['body_len'].mean():.0f} karakter")
print(f"Ortalama özet uzunluğu    : {df_nlp['ozet'].str.len().mean():.0f} karakter")

Yorumlar özetleniyor...


Özetleniyor:   0%|          | 0/200000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (18) is lower than number of sentences (20). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (7) is lower than number of sentences (16). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (8) is lower than number of sentences (10). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of words (5) is lower than number of sentences (6). LSA algorithm may not work properly.
  warn(message % (words_count, sentences_count))
/usr/local/lib/python3.12/dist-packages/sumy/summarizers/lsa.py:76: UserWarning: Number of w

Özetleme tamamlandı
Ortalama orijinal uzunluk : 293 karakter
Ortalama özet uzunluğu    : 139 karakter


In [42]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader_analyzer = SentimentIntensityAnalyzer()

def tam_pipeline(review_body: str, star_rating: int = None) -> dict:

    vader = vader_analyzer.polarity_scores(str(review_body))['compound']

    star_val      = star_rating if star_rating else (1 if vader < -0.3 else 4)
    yildiz_sinyal = int(star_val <= 3)
    vader_sinyal  = int(vader < -0.05)
    problem_skor  = yildiz_sinyal * 0.6 + vader_sinyal * 0.4
    problem_var   = problem_skor >= 0.5

    if problem_var:
        kural_sonuc = kural_ile_departman(review_body)

        if kural_sonuc['guven_skoru'] < 0.45 and 'siniflandirici' in globals():
            try:
                dept_sonuc = zero_shot_departman(review_body)
            except Exception:
                dept_sonuc = kural_sonuc
        else:
            dept_sonuc = kural_sonuc

        departman   = dept_sonuc['departman_tr']
        guven_skoru = dept_sonuc['guven_skoru']
    else:
        departman   = 'Arşiv (Olumlu)'
        guven_skoru = 1.0

    ozet = ozet_cikar(review_body)

    return {
        'ozet':         ozet,
        'problem_var':  problem_var,
        'vader_skoru':  round(vader, 3),
        'problem_skor': round(problem_skor, 2),
        'departman':    departman,
        'guven_skoru':  guven_skoru,
    }

test_yorumlar = [
    ("The screen cracked after one week. Terrible hardware quality.", 1),
    ("App crashes every time I open it. Needs a software fix urgently.", 2),
    ("Package arrived completely destroyed. Box was crushed during shipping.", 1),
    ("Amazing product! Works perfectly and fast delivery. Very happy.", 5),
    ("Customer support never replied to my 5 emails. Unacceptable service.", 2),
]

print("── Tam Pipeline Test Sonuçları ──────────────────────────")
for metin, star in test_yorumlar:
    sonuc = tam_pipeline(metin, star)
    print(f"\nYorum    : {metin[:65]}...")
    print(f"Problem  : {':( Evet' if sonuc['problem_var'] else ':) Hayır'}  "
          f"(skor: {sonuc['problem_skor']})")
    print(f"Departman: {sonuc['departman']}  "
          f"(güven: {sonuc['guven_skoru']:.2f})")
    print(f"Özet     : {sonuc['ozet'][:80]}...")

── Tam Pipeline Test Sonuçları ──────────────────────────

Yorum    : The screen cracked after one week. Terrible hardware quality....
Problem  : :( Evet  (skor: 1.0)
Departman: Teknik Destek  (güven: 0.65)
Özet     : The screen cracked after one week. Terrible hardware quality....

Yorum    : App crashes every time I open it. Needs a software fix urgently....
Problem  : :( Evet  (skor: 0.6)
Departman: Yazılım Ekibi  (güven: 0.75)
Özet     : App crashes every time I open it. Needs a software fix urgently....

Yorum    : Package arrived completely destroyed. Box was crushed during ship...
Problem  : :( Evet  (skor: 1.0)
Departman: Lojistik  (güven: 0.75)
Özet     : Package arrived completely destroyed. Box was crushed during shipping....

Yorum    : Amazing product! Works perfectly and fast delivery. Very happy....
Problem  : :) Hayır  (skor: 0.0)
Departman: Arşiv (Olumlu)  (güven: 1.00)
Özet     : Amazing product! Works perfectly and fast delivery. Very happy....

Yorum    : Customer s

In [45]:
CIKTI = DRIVE_PATH + 'faz3_nlp_sonuc.parquet'
df_nlp.to_parquet(CIKTI, index=False, compression='snappy')

print(f"Kaydedildi: {CIKTI}")
print(f"   Satır : {len(df_nlp):,}")
print(f"   Boyut : {os.path.getsize(CIKTI)/1024**2:.0f} MB")
print(f"\nSonraki adım → Faz4_ML_Modelleme.ipynb")

Kaydedildi: /content/drive/MyDrive/faz3_nlp_sonuc.parquet
   Satır : 200,000
   Boyut : 67 MB

Sonraki adım → Faz4_ML_Modelleme.ipynb


In [49]:
import json

NOTEBOOK_YOLU = '/content/drive/MyDrive/Colab Notebooks/Faz3 NLP Pipeline.ipynb'

with open(NOTEBOOK_YOLU, 'r', encoding='utf-8') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

for cell in nb.get('cells', []):
    cell['metadata'] = {}
    for output in cell.get('outputs', []):
        if 'metadata' in output:
            output['metadata'] = {}

with open(NOTEBOOK_YOLU, 'w', encoding='utf-8') as f:
    json.dump(nb, f, indent=1, ensure_ascii=False)

print("Faz3 temizlendi")

Faz3 temizlendi
